In [2]:
# vector search nb was our ingestion script
# in this nb we need to do the second part: the deployment portion


# load the model 

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# connect to the db

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [4]:
# search 
query = "I just discovered the course. Can I still join it?"

query_vector = model.encode(query)
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [5]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [6]:
# RAG 

# I want to use RAGVector.  dont need to do much because interface  for minsearch and sqlitesearch are similar

from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [7]:
# initialize open ai client

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [9]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [10]:
"""
In this case we split the ingestion and the deployment into 2 separate parts,
and if we need to re-index we can do it here.

this may have been the first and last time i heard about sqlitesearch, 
developed for educational purposes, even while it has some practical usage: it can be deployed for freee in some providers
some providers give the possibility to manage a sqlite db for free 

but in case i need to use some other db i may need to pay more 

thats why he developed sqlitesearch
"""

'\nIn this case we split the ingestion and the deployment into 2 separate parts,\nand if we need to re-index we can do it here.\n\nthis may have been the first and last time i heard about sqlitesearch, \ndeveloped for educational purposes, even while it has some practical usage: it can be deployed for freee in some providers\nsome providers give the possibility to manage a sqlite db for free \n\nbut in case i need to use some other db i may need to pay more \n\nthats why he developed sqlitesearch\n'

In [11]:
# 8 Vector search with PGVector

"""
- extension  for postgres 
- there are other libraries e.g. elasticsearch, special built technologies like quadrant or trauma db ?? , ...
- we'll go with something simple and well known: pg to illustrate
"""


"\n- extension  for postgres \n- there are other libraries e.g. elasticsearch, special built technologies like quadrant or trauma db ?? , ...\n- we'll go with something simple and well known: pg to illustrate\n"

In [12]:
# bring up the pg container

"""
docker run -it \
    --name pgvector \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=pswd \
    -e POSTGRES_DB=faq \
    -v pgvector_data:/var/lib/postgresql/data \
    -p 5432:5432 \
    pgvector/pgvector:pg17
"""



'\ndocker run -it     --name pgvector     -e POSTGRES_USER=user     -e POSTGRES_PASSWORD=pswd     -e POSTGRES_DB=faq     -v pgvector_data:/var/lib/postgresql/data     -p 5432:5432     pgvector/pgvector:pg17\n'

In [13]:
vs_index.close()